# 009 – LoRA Training (Qwen2.5-1.5B)

Fine-tune Qwen2.5-1.5B with LoRA adapters for AI text detection.
Uses the adapters library (AdapterHub) and logs to W&B.

## 1. Setup

In [5]:
# Install dependencies (run once)
# !pip install -q transformers adapters datasets torch scikit-learn accelerate wandb python-dotenv huggingface_hub dotenv

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Load API keys from root .env
load_dotenv(Path("../../../.env"))

# Add src to path
SRC_DIR = Path("../../src/009-lora-replication")
sys.path.insert(0, str(SRC_DIR))

from config import (
    TRAIN_FILE, VAL_FILE, ARTIFACTS_DIR, ADAPTER_NAME,
    WANDB_PROJECT, WANDB_ENTITY, WANDB_RUN_NAME,
)
from data import load_jsonl, get_tokenizer, tokenize_dataset
from model import create_lora_model
from train import get_training_args, run_training

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

ModuleNotFoundError: No module named 'config'

In [ ]:
import wandb
from huggingface_hub import login as hf_login

# Authenticate using keys from .env
wandb.login(key=os.environ["WANDB_API_KEY"])
hf_login(token=os.environ["HF_TOKEN"])

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
os.environ["WANDB_ENTITY"] = WANDB_ENTITY

## 2. Load and Tokenize Data

In [ ]:
tokenizer = get_tokenizer()

train_df = load_jsonl(str(TRAIN_FILE))
val_df = load_jsonl(str(VAL_FILE))
print(f"Train: {len(train_df)} samples")
print(f"Val:   {len(val_df)} samples")

In [ ]:
train_dataset = tokenize_dataset(train_df, tokenizer)
val_dataset = tokenize_dataset(val_df, tokenizer)
print(f"Train: {train_dataset}")
print(f"Val:   {val_dataset}")

## 3. Create Model

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = create_lora_model(device=device)

## 4. Train

In [ ]:
output_dir = str(ARTIFACTS_DIR / "checkpoints")
training_args = get_training_args(output_dir, use_cpu=(device == "cpu"))

trainer, eval_results = run_training(
    model, tokenizer, train_dataset, val_dataset, training_args
)

## 5. Save Adapter & Upload to HuggingFace

In [ ]:
import json

# Save adapter locally
adapter_dir = ARTIFACTS_DIR / "lora_adapter"
model.save_adapter(str(adapter_dir), ADAPTER_NAME)
tokenizer.save_pretrained(str(adapter_dir))
print(f"Adapter saved to {adapter_dir}")

# Save training metrics
with open(ARTIFACTS_DIR / "training_val_metrics.json", "w") as f:
    json.dump(eval_results, f, indent=2)
print("Metrics saved.")

In [ ]:
# Upload adapter to HuggingFace
from huggingface_hub import HfApi

HF_REPO = "hersheys-baklava/qwen-lora-pan2026-009"

api = HfApi()
api.create_repo(HF_REPO, exist_ok=True)
api.upload_folder(
    folder_path=str(adapter_dir),
    repo_id=HF_REPO,
    repo_type="model",
)
print(f"Adapter uploaded to https://huggingface.co/{HF_REPO}")

In [ ]:
wandb.finish()